# **Installing Required Packages**

In [ ]:
!pip install langchain-groq langchain langchain-community

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install google-search-results

In [ ]:
!pip install langchain_classic langchain-community

# **Loading API keys from Colab Secrets**

In [ ]:
!unzip /content/data_store.zip

## **LangGraph**

## A Simple LLM Wrapper

In [ ]:
# minimal_langgraph_gemini.py
import os
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

# -------- State --------
class State(BaseModel):
    input: str = Field(...)
    output: str | None = None

# -------- Gemini LLM --------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

# -------- Node --------
def respond(state: State) -> State:
    raw = llm.invoke(state.input)
    # minimal normalization without helpers
    if hasattr(raw, "content"):
        text = raw.content
    else:
        text = str(raw)
    return State(input=state.input, output=text)

# -------- Build Graph --------
graph = StateGraph(State)
graph.add_node("respond", respond)
graph.set_entry_point("respond")
graph.add_edge("respond", END)
app = graph.compile()

# -------- Run --------
result = app.invoke({"input": "Explain LangGraph in one line"})
print(result['output'])


## A Mock Question Answering

In [ ]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

class State(BaseModel):
    query: str = Field(...)
    action: str | None = None
    docs: list | None = None
    answer: str | None = None

# mock db & llm as above
class MockDB:
    def similarity_search(self, q):
        return [type("D", (), {"page_content": "Product X released in 2010."})()]

class MockLLM:
    def invoke(self, prompt):
        return f"ANSWER_FROM_LLM: {prompt[:80]}..."

db = MockDB()
llm = MockLLM()

def planner(state: State):
    q = state.query
    state.action = "retrieve" if ("year" in q or "released" in q) else "answer_direct"
    return state

def retrieve(state: State):
    state.docs = db.similarity_search(state.query)
    return state

def answer(state: State):
    context = "\n".join([d.page_content for d in (state.docs or [])])
    prompt = f"Context:\n{context}\n\nQuestion:\n{state.query}"
    raw = llm.invoke(prompt)
    text = getattr(raw, "content", raw)
    state.answer = str(text)
    return state

graph = StateGraph(State)
graph.add_node("planner", planner)
graph.add_node("retriever", retrieve)
graph.add_node("answer", answer)

graph.set_entry_point("planner")
graph.add_conditional_edges("planner",
    lambda s: s.action,
    {"retrieve": "retriever", "answer_direct": "answer"}
)
graph.add_edge("retriever", "answer")
graph.add_edge("answer", END)

app = graph.compile()

# Must pass 'query' when invoking because Pydantic requires it
result = app.invoke({"query": "When was Product X released?"})

print("answer:", result['answer'])


## **Multi Agent Systems**

In [ ]:
"""
RAG pipeline — Sherlock Holmes corpus
--------------------------------------
Embeddings : TF-IDF + LSA (TruncatedSVD) -> dense float32
Vector DB  : FAISS  (IndexFlatIP  = cosine on L2-normalised vectors)
LLM        : Groq API  (llama-3.3-70b-versatile)
Graph      : LangGraph StateGraph  (planner -> retriever? -> answer)

Setup
-----
    pip install groq faiss-cpu langgraph pydantic scikit-learn
    export GROQ_API_KEY=gsk_...
    python rag_sherlock.py
"""

import os
import pickle
import textwrap
from pathlib import Path
from typing import Optional

import numpy as np
import faiss
from groq import Groq
from pydantic import BaseModel, Field
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from langgraph.graph import StateGraph, END

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1. CONSTANTS
# ─────────────────────────────────────────────────────────────────

DATA_DIR    = Path("data_store")
CACHE_FILE  = Path("faiss_store.pkl")
EMBED_DIM   = 256     # LSA components (auto-capped to vocab size)
CHUNK_SIZE  = 400     # words per chunk
CHUNK_STEP  = 200     # stride  (50 % overlap)
TOP_K       = 4       # passages retrieved per query

BOOK_NAMES = {
    "advs": "The Adventures of Sherlock Holmes",
    "case": "The Case-Book of Sherlock Holmes",
    "lstb": "His Last Bow",
    "mems": "The Memoirs of Sherlock Holmes",
    "retn": "The Return of Sherlock Holmes",
}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2. CORPUS LOADING & CHUNKING
# ─────────────────────────────────────────────────────────────────

def load_corpus() -> tuple[list[str], list[dict]]:
    chunks, meta = [], []
    files = sorted(DATA_DIR.glob("*.txt"))
    for path in files:
        book_name = BOOK_NAMES.get(path.stem, path.stem)
        words = path.read_text(encoding="utf-8", errors="ignore").split()
        for i in range(0, len(words), CHUNK_STEP):
            chunk = " ".join(words[i : i + CHUNK_SIZE])
            if len(chunk.strip()) < 80:
                continue
            chunks.append(chunk)
            meta.append({"book": book_name, "offset": i})
    print(f"[corpus]  {len(chunks):,} chunks from {len(files)} books")
    return chunks, meta



In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3. FAISS VECTOR STORE
# ─────────────────────────────────────────────────────────────────

class FAISSVectorStore:
    """
    Builds a FAISS IndexFlatIP index from TF-IDF + LSA embeddings.
    The index and pipeline are persisted to CACHE_FILE so the
    expensive fit step only runs once.
    """

    def __init__(self):
        if CACHE_FILE.exists():
            print("[faiss]  loading index from cache ...")
            with open(CACHE_FILE, "rb") as f:
                data = pickle.load(f)
            self.vectorizer: TfidfVectorizer = data["vectorizer"]
            self.svd:        TruncatedSVD    = data["svd"]
            self.dim:        int             = data["dim"]
            self.chunks:     list[str]       = data["chunks"]
            self.meta:       list[dict]      = data["meta"]
            # Rebuild FAISS index from stored vectors
            vecs = data["vectors"]               # (N, dim) float32
            self.index = faiss.IndexFlatIP(self.dim)
            self.index.add(vecs)
            print(f"[faiss]  {self.index.ntotal:,} vectors  dim={self.dim}")
        else:
            self._build()

    # ------------------------------------------------------------------
    def _build(self):
        self.chunks, self.meta = load_corpus()

        print("[faiss]  fitting TF-IDF ...")
        self.vectorizer = TfidfVectorizer(
            max_features=80_000,
            ngram_range=(1, 2),
            sublinear_tf=True,
        )
        X_sparse = self.vectorizer.fit_transform(self.chunks)

        # Cap LSA components to what the vocabulary allows
        max_comp   = min(EMBED_DIM, X_sparse.shape[1] - 1, X_sparse.shape[0] - 1)
        self.dim   = max_comp
        print(f"[faiss]  fitting LSA  dim={self.dim} ...")
        self.svd   = TruncatedSVD(n_components=self.dim, random_state=42)
        X_dense    = self.svd.fit_transform(X_sparse).astype(np.float32)
        faiss.normalize_L2(X_dense)              # unit vectors -> cosine via IP

        print("[faiss]  building index ...")
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(X_dense)

        with open(CACHE_FILE, "wb") as f:
            pickle.dump({
                "vectorizer": self.vectorizer,
                "svd":        self.svd,
                "dim":        self.dim,
                "chunks":     self.chunks,
                "meta":       self.meta,
                "vectors":    X_dense,
            }, f)
        print(f"[faiss]  index built  {self.index.ntotal:,} vectors  dim={self.dim}")

    # ------------------------------------------------------------------
    def _embed(self, text: str) -> np.ndarray:
        """Return a (1, dim) L2-normalised float32 query vector."""
        x = self.svd.transform(
            self.vectorizer.transform([text])
        ).astype(np.float32)
        faiss.normalize_L2(x)
        return x

    def similarity_search(self, query: str, k: int = TOP_K) -> list[dict]:
        q_vec = self._embed(query)
        scores, indices = self.index.search(q_vec, k)
        return [
            {
                "page_content": self.chunks[idx],
                "score":        float(scores[0][rank]),
                "book":         self.meta[idx]["book"],
                "offset":       self.meta[idx]["offset"],
            }
            for rank, idx in enumerate(indices[0])
            if idx != -1
        ]

In [ ]:
# ─────────────────────────────────────────────────────────────────# ─────────────────────────────────────────────────────────────────
from google.colab import userdata

# 4. GROQ LLM

class GroqLLM:
    def __init__(self, model: str = "llama-3.3-70b-versatile"):
        api_key = userdata.get('GROQ_API_KEY')
        if not api_key:
            raise EnvironmentError(
                "GROQ_API_KEY is not set.\n"
                "Export it before running:  export GROQ_API_KEY=gsk_..."
            )
        self.client = Groq(api_key=api_key)
        self.model  = model

    def invoke(self, system: str, user: str, max_tokens: int = 512) -> str:
        response = self.client.chat.completions.create(
            model      = self.model,
            max_tokens = max_tokens,
            messages   = [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
        )
        return response.choices[0].message.content.strip()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5. LANGGRAPH STATE & NODES
# ─────────────────────────────────────────────────────────────────

class State(BaseModel):
    query:  str            = Field(...)
    action: Optional[str]  = None
    docs:   Optional[list] = None
    answer: Optional[str]  = None


# Initialise vector store at import time; LLM is lazy.
db   = FAISSVectorStore()
_llm: Optional[GroqLLM] = None

def get_llm() -> GroqLLM:
    global _llm
    if _llm is None:
        _llm = GroqLLM()
    return _llm


PLANNER_SYSTEM = textwrap.dedent("""\
    You are a routing assistant for a Sherlock Holmes Q&A system.
    Decide whether the question needs searching the books for specific
    plot/character/event details (action = "retrieve") or can be
    answered from general knowledge (action = "answer_direct").

    Reply with ONLY one of these two words, nothing else:
      retrieve
      answer_direct
""")

ANSWER_SYSTEM = textwrap.dedent("""\
    You are an expert on Arthur Conan Doyle's Sherlock Holmes stories.
    Answer the user's question concisely and accurately, using the
    provided context passages as your primary source.
    Mention which story or book the information comes from when possible.
    If the context is insufficient, say so honestly.
""")


def planner(state: State) -> State:
    decision     = get_llm().invoke(PLANNER_SYSTEM, state.query, max_tokens=8)
    state.action = "retrieve" if "retrieve" in decision.lower() else "answer_direct"
    print(f"[planner]   action = {state.action}")
    return state


def retrieve(state: State) -> State:
    results    = db.similarity_search(state.query, k=TOP_K)
    state.docs = results
    print(f"[retriever] top hit  score={results[0]['score']:.3f}  | {results[0]['book']}")
    return state


def answer(state: State) -> State:
    if state.docs:
        passages = "\n\n".join(
            f"[Passage {i} - {d['book']}]\n{d['page_content']}"
            for i, d in enumerate(state.docs, 1)
        )
        user_msg = f"Context:\n{passages}\n\nQuestion:\n{state.query}"
    else:
        user_msg = state.query

    state.answer = get_llm().invoke(ANSWER_SYSTEM, user_msg, max_tokens=512)
    return state



In [ ]:
# ─────────────────────────────────────────────────────────────────
# 6. BUILD GRAPH
# ─────────────────────────────────────────────────────────────────

graph = StateGraph(State)
graph.add_node("planner",   planner)
graph.add_node("retriever", retrieve)
graph.add_node("answer",    answer)

graph.set_entry_point("planner")
graph.add_conditional_edges(
    "planner",
    lambda s: s.action,
    {"retrieve": "retriever", "answer_direct": "answer"},
)
graph.add_edge("retriever", "answer")
graph.add_edge("answer",    END)

app = graph.compile()


# ─────────────────────────────────────────────────────────────────
# 7. PUBLIC API
# ─────────────────────────────────────────────────────────────────

def ask(question: str) -> str:
    """Run the full RAG pipeline and return the answer string."""
    result = app.invoke({"query": question})
    return result["answer"]


# ─────────────────────────────────────────────────────────────────
# 8. CLI DEMO
# ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    questions = [
        "Who is Irene Adler and how does she outwit Holmes?",
        "What disguises does Sherlock Holmes use across the stories?",
        "Describe the final confrontation between Holmes and Moriarty.",
        "What is Watson's role in the investigations?",
        "Who wrote the Sherlock Holmes stories?",    # -> answer_direct
    ]

    print("\n" + "=" * 72)
    for q in questions:
        print(f"\nQ: {q}")
        print(f"A: {ask(q)}")
        print("-" * 72)